In [ ]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, expr

dp.create_streaming_table(
    name="shipments_silver",
    comment="Full shipment history using AUTO CDC SCD Type 2",
    table_properties={
        "delta.enableChangeDataFeed": "true"
    }
)

dp.create_auto_cdc_flow(
    target="shipments_silver",
    source="shipments_bronze",
    keys=["shipment_id"],
    sequence_by=col("_source_version"),
    apply_as_deletes=expr("_operation = 'DELETE'"),
    stored_as_scd_type="2",
    # Ingest metadata churns on every event; only business attribute changes
    # should open a new history row.
    track_history_except_column_list=[
        "_operation",
        "_source_file",
        "_ingest_ts",
        "_rescued_data",
        "_source_updated_ts",
    ],
)